# MMELON Fine-Tuning with K-Fold Cross-Validation

Fine-tune the pretrained MMELON model (`ibm-research/biomed.sm.mv-te-84m`) for **binary classification**
on every fold of a k-fold split, and select the final model by the **average metric across folds**.

**Environment:** run with the `biomed-multiview` kernel.

**Inputs**
- A CSV with a SMILES column and a 0/1 label column.
- An `.npz` file of fold indices from [cluster_cv_folds.ipynb](cluster_cv_folds.ipynb) (or [stratified_cv.py](stratified_cv.py))
  (arrays `fold0_train`, `fold0_test`, `fold1_train`, ...). The number of folds is read from this file.

**Metrics**
- **hit@K** (hit@200, hit@100, hit@50): rank the test molecules by predicted score; hit@K is how many
  of the top K have label 1. If a test set has fewer than K molecules, all of them are counted.
- Standard classification metrics: AUROC, precision, recall, specificity and loss.

**How the model is selected**
1. Train all folds side by side, one epoch at a time (up to `EPOCHS`), scoring each fold's test rows after
   every epoch.
2. After each epoch, average the metrics over all folds.
3. Rank epochs by `SELECT_METRICS` (default: hit@200, then hit@100, then hit@50, then AUROC to break ties).
   Each fold's model from the best epoch is kept (K models, usable as an ensemble), and only the best is
   stored on disk. Training stops early when `PATIENCE` epochs in a row bring no improvement.

Because the test folds are used to pick the epoch, the selected average is slightly optimistic.
For an unbiased estimate, evaluate the final models on data not in the CSV.

**Output** (in the gitignored `aircheck/output/`):
```
output/
├── base_models/biomed.sm.mv-te-84m/      # pretrained model (downloaded once)
└── <DATASET>/<RUN_NAME>/
    ├── config.json                       # settings used
    ├── progress.log                      # one line per fold-epoch and per round, written while training
    ├── cv_metrics_per_epoch.csv          # every fold × epoch
    ├── cv_summary_per_epoch.csv          # mean and std over folds per epoch
    ├── selection.json                    # selected epoch, its scores, model paths
    └── fold_0/, fold_1/, ...
        ├── data/                         # data_train.csv, data_val.csv (= this fold's test rows)
        ├── metrics_per_epoch.csv         # this fold's test metrics per epoch
        └── best_model.ckpt               # this fold's model at the selected epoch
```

## 1. Inputs and settings

**Edit these before running.** Data files can be `.csv` or `.parquet`; only the columns the notebook needs are read.

In [1]:
from pathlib import Path

# Dataset name: all outputs go under aircheck/output/<DATASET>/ (the pretrained base model is shared)
DATASET = "sample-data"

# Name of this run; outputs go to output/<DATASET>/<RUN_NAME>/ (must not already exist)
RUN_NAME = "mmelon-cv"

# ── Data ──────────────────────────────────────────────────────────────────────
DATA_DIR = Path("../output") / DATASET / "cv_folds"   # made by cluster_cv_folds.ipynb

DATA_PATH = DATA_DIR / "train_cluster_cv_data.csv"
CV_INDICES_PATH = DATA_DIR / "train_cluster_cv_folds.npz"   # the number of folds is read from it

SMILES_COL = "SMILES"
LABEL_COL = "LABEL"

In [2]:
import os

# ── Model ─────────────────────────────────────────────────────────────────────
BASE_MODEL = "ibm-research/biomed.sm.mv-te-84m"  # Hugging Face model ID

# ── Training ──────────────────────────────────────────────────────────────────
EPOCHS = 50               # maximum epochs; all folds train side by side and stop together
PATIENCE = 5              # stop after this many epochs without improvement of the fold average
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
SEED = 42

# Balance the loss between the classes: label 0 gets weight 1, label 1 gets (number of 0s) / (number of 1s),
# counted on the training data only (in CV: on each fold's own training rows). False = unweighted loss.
BALANCE_CLASSES = True

# hit@K = number of label-1 molecules among the K highest-scored test molecules
HIT_KS = [200, 100, 50]

# Model selection, in priority order: the first metric decides, later ones break ties.
# Choose from hit@K (for K in HIT_KS), "rocauc", "precision", "recall", "specificity", "loss".
SELECT_METRICS = ["hit@200", "hit@100", "hit@50", "rocauc"]

NUM_WORKERS = min(24, os.cpu_count())  # data-loading processes: all CPUs, up to 24

In [3]:
import json
import shutil

import numpy as np
import pandas as pd
import torch


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bmfm_sm").is_dir() and (path / "pyproject.toml").exists():
            return path
    raise FileNotFoundError(f"Run this notebook from inside the biomed-multi-view repo (cwd={start})")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
OUTPUT_DIR = REPO_ROOT / "aircheck" / "output"
BASE_MODEL_DIR = OUTPUT_DIR / "base_models" / BASE_MODEL.split("/")[-1]
RUN_DIR = OUTPUT_DIR / DATASET / RUN_NAME

for path in (DATA_PATH, CV_INDICES_PATH):
    assert Path(path).exists(), f"File not found: {path}"
assert not (RUN_DIR.exists() and any(RUN_DIR.iterdir())), (
    f"{RUN_DIR} already exists. Choose a new RUN_NAME or delete that folder."
)

# ── Fold indices ──────────────────────────────────────────────────────────────
cv_indices = np.load(CV_INDICES_PATH)
N_FOLDS = len([k for k in cv_indices.files if k.endswith("_train")])
folds = [(cv_indices[f"fold{i}_train"], cv_indices[f"fold{i}_test"]) for i in range(N_FOLDS)]
for i, (train_idx, test_idx) in enumerate(folds):
    assert not np.intersect1d(train_idx, test_idx).size, f"fold {i}: train and test rows overlap"

# Each fold keeps its latest training state (weights + optimizer, ~1 GB) and its best model (~0.35 GB)
needed_gb = 1.4 * N_FOLDS
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
free_gb = shutil.disk_usage(OUTPUT_DIR).free / 1e9
assert free_gb > needed_gb + 2, f"Need ~{needed_gb:.0f} GB free for checkpoints, only {free_gb:.0f} GB available"

RUN_DIR.mkdir(parents=True, exist_ok=True)
config = dict(
    data_path=str(DATA_PATH), cv_indices_path=str(CV_INDICES_PATH), n_folds=N_FOLDS,
    smiles_col=SMILES_COL, label_col=LABEL_COL, base_model=BASE_MODEL, epochs=EPOCHS, patience=PATIENCE,
    batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
    seed=SEED, hit_ks=HIT_KS, select_metrics=SELECT_METRICS, balance_classes=BALANCE_CLASSES,
)
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))

print(f"Folds      : {N_FOLDS} (from {CV_INDICES_PATH.name})")
print(f"Run folder : {RUN_DIR}")
print(f"Disk       : ~{needed_gb:.1f} GB needed during training, {free_gb:.0f} GB free")
print(f"Device     : {'GPU' if torch.cuda.is_available() else 'CPU (training will be slow)'}")

Folds      : 1 (from train_cluster_cv_folds.npz)
Run folder : ~/repos/biomed-multi-view/aircheck/output/sample-data/mmelon-cv
Disk       : ~1.4 GB needed during training, 38 GB free
Device     : GPU


## 2. Prepare data

Loads the CSV and writes each fold's rows in the format the MMELON data pipeline expects
(`data_train.csv`, and `data_val.csv` for the rows scored after each epoch).

In [4]:
import pyarrow.parquet as pq


def read_table(path, columns, optional=()) -> pd.DataFrame:
    # Read only the needed columns from a .csv or .parquet file; optional columns are read if present
    path = Path(path)
    is_parquet = path.suffix.lower() in (".parquet", ".pq")
    available = pq.ParquetFile(path).schema_arrow.names if is_parquet else pd.read_csv(path, nrows=0).columns.tolist()
    missing = [c for c in columns if c not in available]
    assert not missing, f"{path.name} has no column(s) {missing}; available: {available}"
    wanted = list(dict.fromkeys([*columns, *(c for c in optional if c in available)]))
    df = pd.read_parquet(path, columns=wanted) if is_parquet else pd.read_csv(path, usecols=wanted)
    return df[wanted].reset_index(drop=True)


def class_weights_for(labels: pd.Series):
    # Loss weights [[label 0, label 1]] from the training labels, or None (unweighted) when BALANCE_CLASSES is off
    if not BALANCE_CLASSES:
        return None
    n_neg, n_pos = int((labels == 0).sum()), int((labels == 1).sum())
    assert n_neg > 0 and n_pos > 0, "training data needs both labels to balance the classes"
    return [[1.0, round(n_neg / n_pos, 4)]]


df = read_table(DATA_PATH, [SMILES_COL, LABEL_COL]).rename(columns={SMILES_COL: "smiles", LABEL_COL: "label"})

# Rows are referenced by position, so bad rows are reported rather than dropped
bad = df["smiles"].isna() | (df["smiles"].astype(str).str.strip() == "") | df["label"].isna()
assert not bad.any(), f"{bad.sum()} rows have empty SMILES or label, e.g. rows {df.index[bad][:5].tolist()}"
df["label"] = df["label"].astype(int)
assert set(df["label"].unique()) <= {0, 1}, "labels must be 0/1"

max_index = max(int(idx.max()) for pair in folds for idx in pair)
assert max_index < len(df), f"Fold indices go up to {max_index} but {DATA_PATH.name} has {len(df)} rows"

summary, CLASS_WEIGHTS = [], []
for i, (train_idx, test_idx) in enumerate(folds):
    data_dir = RUN_DIR / f"fold_{i}" / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    df.iloc[train_idx].to_csv(data_dir / "data_train.csv", index=False)
    df.iloc[test_idx].to_csv(data_dir / "data_val.csv", index=False)
    CLASS_WEIGHTS.append(class_weights_for(df["label"].iloc[train_idx]))
    summary.append(dict(
        fold=i, train=len(train_idx), test=len(test_idx),
        train_pos_pct=round(100 * df["label"].iloc[train_idx].mean(), 2),
        test_pos_pct=round(100 * df["label"].iloc[test_idx].mean(), 2),
        class_weights=CLASS_WEIGHTS[i],
    ))

print(f"{len(df):,} rows, {100 * df['label'].mean():.2f}% positive")
config["class_weights_per_fold"] = CLASS_WEIGHTS
(RUN_DIR / "config.json").write_text(json.dumps(config, indent=2))
pd.DataFrame(summary)

10,221 rows, 2.16% positive


,fold,train,test,train_pos_pct,test_pos_pct,class_weights
0,0,7590,2600,1.19,3.85,"[[1.0, 83.3333]]"


## 3. Download the base model

Downloaded once into `output/base_models/`; later runs reuse it.

In [5]:
from huggingface_hub import snapshot_download

if (BASE_MODEL_DIR / "model.safetensors").exists():
    print(f"Using existing base model: {BASE_MODEL_DIR}")
else:
    print(f"Downloading {BASE_MODEL} ...")
    snapshot_download(repo_id=BASE_MODEL, local_dir=BASE_MODEL_DIR)
    print(f"Saved to: {BASE_MODEL_DIR}")

Using existing base model: ~/repos/biomed-multi-view/aircheck/output/base_models/biomed.sm.mv-te-84m


## 4. Model and data helpers

In [6]:
import gc
import logging
import warnings

warnings.filterwarnings("ignore")
logging.getLogger().setLevel(logging.WARNING)

import pytorch_lightning as pl
from torch.utils.data import DataLoader

from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel
from bmfm_sm.core.data_modules.namespace import LateFusionStrategy, TaskType
from bmfm_sm.predictive.data_modules.multimodal_finetune_dataset import MultiModalFinetuneDataPipeline
from bmfm_sm.predictive.modules.finetune_lightning_module import FineTuneLightningModule

FUSION_STRATEGY = LateFusionStrategy.ATTENTIONAL

dataset_args = {
    "task_type": TaskType.CLASSIFICATION,
    "num_tasks": 1,
    "modalities": ["TEXT_MODEL", "IMAGE_MODEL", "GRAPH_2D_MODEL"],
    "smiles_col": "smiles",
    "label_cols": ["label"],
    "split_col": None,
}

# Load the pretrained weights once; every fold starts from a copy of them
pretrained_state = SmallMoleculeMultiViewModel.from_pretrained(
    FUSION_STRATEGY, model_path=str(BASE_MODEL_DIR), huggingface=True
).state_dict()


def make_loader(data_dir: Path, stage: str, shuffle: bool) -> DataLoader:
    dataset = MultiModalFinetuneDataPipeline(data_dir=str(data_dir), dataset_args=dataset_args, stage=stage)
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle,
                      collate_fn=dataset.collate_fn, num_workers=NUM_WORKERS)


class HitAtKFineTuneModule(FineTuneLightningModule):
    # Adds hit@K to the scores logged after every epoch (as val_hit@K)

    def on_validation_epoch_start(self):
        super().on_validation_epoch_start()
        self._val_scores, self._val_labels = [], []

    def validation_step(self, batch, batch_idx):
        loss, y, y_hat = self._shared_step(batch, "val")
        self._val_scores.append(y_hat.detach().float().reshape(-1).cpu())
        self._val_labels.append(y.detach().reshape(-1).cpu())
        return loss

    def on_validation_epoch_end(self):
        super().on_validation_epoch_end()
        scores, labels = torch.cat(self._val_scores), torch.cat(self._val_labels)
        top = torch.argsort(scores, descending=True)
        for k in HIT_KS:
            self.log(f"val_hit@{k}", float((labels[top[:k]] == 1).sum()))


def make_model(fold: int) -> HitAtKFineTuneModule:
    module = HitAtKFineTuneModule(
        base_model_class="bmfm_sm.predictive.modules.smmv_model.SmallMoleculeMultiView",
        model_params={
            "agg_arch": FUSION_STRATEGY.value[0],
            "agg_gate_input": FUSION_STRATEGY.value[1],
            "agg_weight_freeze": FUSION_STRATEGY.value[2],
            "inference_mode": False,
        },
        task_type="classification",
        num_tasks=1,
        checkpoint_path=None,
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        finetuning_args={
            "weight_freeze": "unfrozen",
            "initialization": "default",
            "head_arch": "mlp",
            "use_norm": True,
            "head_dropout": 0.2,
            "class_weights": CLASS_WEIGHTS[fold],
        },
    )
    result = module.model.load_state_dict(pretrained_state, strict=False)
    # strict=False would silently skip mismatched weights, so check explicitly
    assert not result.missing_keys and not result.unexpected_keys, (
        f"Pretrained weights did not match: missing={result.missing_keys[:5]}, "
        f"unexpected={result.unexpected_keys[:5]}"
    )
    return module


def order_metrics(columns) -> list:
    # hit@K first (in HIT_KS order), then the standard metrics
    hits = [f"hit@{k}" for k in HIT_KS]
    return [c for c in hits if c in columns] + [c for c in columns if c not in hits]


print("Helpers ready; pretrained weights loaded.")

2026-10-20 15:19:42,036 - rdkit - INFO - instance-shay-ml-01:140262262728512:0:0 - Enabling RDKit 2026.09.1 jupyter extensions


2026-10-20 15:19:42,223 - root - INFO - instance-shay-ml-01:140262262728512:0:0 - Loading checkpoint via HuggingFace Hub from provided path ~/repos/biomed-multi-view/aircheck/output/base_models/biomed.sm.mv-te-84m


2026-10-20 15:19:43,618 - root - INFO - instance-shay-ml-01:140262262728512:0:0 - Using coeff_mlp architecture for aggregator


2026-10-20 15:19:43,619 - root - INFO - instance-shay-ml-01:140262262728512:0:0 - dim_list [512, 512, 768] for aggregator


2026-10-20 15:19:43,749 - root - INFO - instance-shay-ml-01:140262262728512:0:0 - in train False setting deterministic_eval = True


Loading weights from local directory
Helpers ready; pretrained weights loaded.


## 5. Train the folds side by side

The folds train in rounds of one epoch: epoch 0 of every fold, then epoch 1 of every fold, and so on. After each
round, the fold-averaged metrics are compared with the best round so far by `SELECT_METRICS` (the first metric
decides, later ones break ties). When a round is the best so far, every fold's model from that round is saved as
`fold_<i>/best_model.ckpt`, replacing the previous best. Training stops early when `PATIENCE` rounds in a row bring
no improvement. Only these best models and each fold's latest training state are kept on disk.
Progress is printed after every fold-epoch and appended to `progress.log` in the run folder (follow it with
`tail -f`); the per-epoch metrics files are updated after every round, so they survive an interrupted run.
Metrics prefixed `val_` are scores on each fold's **test** rows.

In [7]:
%%time
import time

allowed = [f"hit@{k}" for k in HIT_KS] + ["rocauc", "precision", "recall", "specificity", "loss"]
assert set(SELECT_METRICS) <= set(allowed), f"SELECT_METRICS must come from {allowed}"


def round_key(means) -> tuple:
    # Larger is better; loss is negated so a lower loss ranks higher. Ties keep the earlier epoch.
    return tuple(float(means[m]) * (-1 if m == "loss" else 1) for m in SELECT_METRICS)


def save_weights(state_path: Path, out_path: Path):
    # The full training state without the optimizer: loadable with FineTuneLightningModule.load_from_checkpoint
    ckpt = torch.load(state_path, map_location="cpu")
    for key in ("optimizer_states", "lr_schedulers"):
        ckpt.pop(key, None)
    torch.save(ckpt, out_path)


loaders = {
    fold: (make_loader(RUN_DIR / f"fold_{fold}" / "data", "train", shuffle=True),
           make_loader(RUN_DIR / f"fold_{fold}" / "data", "val", shuffle=False))
    for fold in range(N_FOLDS)
}
rows, best_key, BEST_EPOCH, wait = [], None, None, 0
LOG = RUN_DIR / "progress.log"  # follow a running job with: tail -f <run folder>/progress.log


def log(message: str):
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {message}"
    print(line, flush=True)
    with open(LOG, "a") as f:
        f.write(line + "\n")


def save_metrics(rows):
    # Write the per-epoch metrics so far, so they survive an interrupted run
    df = pd.DataFrame(rows)
    names = order_metrics([c for c in df.columns if c not in ("fold", "epoch")])
    df = df[["fold", "epoch"] + names]
    df.to_csv(RUN_DIR / "cv_metrics_per_epoch.csv", index=False)
    for f in range(N_FOLDS):
        df[df["fold"] == f].to_csv(RUN_DIR / f"fold_{f}" / "metrics_per_epoch.csv", index=False)
    return df, names


log(f"Training {N_FOLDS} folds side by side, up to {EPOCHS} epochs (patience {PATIENCE})")

for epoch in range(EPOCHS):
    for fold in range(N_FOLDS):
        fold_dir = RUN_DIR / f"fold_{fold}"
        state_path = fold_dir / "training_state.ckpt"
        pl.seed_everything(SEED + 1000 * fold + epoch, verbose=False)
        start = time.time()
        trainer = pl.Trainer(
            max_epochs=epoch + 1, logger=False, enable_checkpointing=False, default_root_dir=fold_dir,
            accelerator="auto", devices=1, num_sanity_val_steps=0,
            enable_model_summary=False, enable_progress_bar=False,
        )
        module = make_model(fold)
        # Continue this fold from its previous epoch (weights and optimizer state are restored)
        trainer.fit(module, *loaders[fold], ckpt_path=state_path if epoch > 0 else None)
        trainer.save_checkpoint(state_path)
        rows.append(dict(fold=fold, epoch=epoch, **{
            k.removeprefix("val_"): float(v) for k, v in trainer.callback_metrics.items() if k.startswith("val_")
        }))
        log(f"epoch {epoch}, fold {fold + 1}/{N_FOLDS} done in {(time.time() - start) / 60:.1f} min")
        del trainer, module
        gc.collect()

    cv_metrics = pd.DataFrame(rows)
    means = cv_metrics[cv_metrics["epoch"] == epoch].mean(numeric_only=True)
    key = round_key(means)
    if best_key is None or key > best_key:
        best_key, BEST_EPOCH, wait = key, epoch, 0
        for fold in range(N_FOLDS):
            save_weights(RUN_DIR / f"fold_{fold}" / "training_state.ckpt", RUN_DIR / f"fold_{fold}" / "best_model.ckpt")
        status = "best so far, models saved"
    else:
        wait += 1
        status = f"no improvement ({wait}/{PATIENCE})"
    log(f"epoch {epoch:3d} mean over folds:  " + "  ".join(f"{m} {means[m]:.4f}" for m in SELECT_METRICS)
        + f"  -> {status}")
    save_metrics(rows)
    if wait >= PATIENCE:
        log(f"Stopping early: no improvement of the fold average for {PATIENCE} epochs")
        break

for fold in range(N_FOLDS):
    (RUN_DIR / f"fold_{fold}" / "training_state.ckpt").unlink()
EPOCHS_RUN = epoch + 1

cv_metrics, metric_names = save_metrics(rows)
log(f"Done: selected epoch {BEST_EPOCH} after {EPOCHS_RUN} epochs")


2026-10-20 15:19:43,777 - root - INFO - instance-shay-ml-01:140262262728512:0:0 - Working on task TaskType.CLASSIFICATION


2026-10-20 15:19:43,778 - root - INFO - instance-shay-ml-01:140262262728512:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


2026-10-20 15:19:46,342 - root - INFO - instance-shay-ml-01:140262262728512:0:0 - Working on task TaskType.CLASSIFICATION


2026-10-20 15:19:46,343 - root - INFO - instance-shay-ml-01:140262262728512:0:0 - modalities running in data pipeline TEXT_MODEL,IMAGE_MODEL,GRAPH_2D_MODEL


2026-10-20 15:19:48  Training 1 folds side by side, up to 50 epochs (patience 5)


...

2026-10-20 15:52:21  epoch 10, fold 1/1 done in 3.0 min


2026-10-20 15:52:21  epoch  10 mean over folds:  hit@200 32.0000  hit@100 17.0000  hit@50 9.0000  rocauc 0.7589  -> no improvement (5/5)


2026-10-20 15:52:21  Stopping early: no improvement of the fold average for 5 epochs


2026-10-20 15:52:21  Done: selected epoch 5 after 11 epochs


CPU times: user 30min 4s, sys: 1min 39s, total: 31min 44s
Wall time: 32min 37s


## 6. Selected epoch

The epoch was chosen during training (step 5). This table ranks every epoch by the fold-averaged
`SELECT_METRICS` as a check, and shows each fold's scores at the selected epoch.

In [8]:
cv_summary = cv_metrics.groupby("epoch")[metric_names].agg(["mean", "std"])
cv_summary.columns = [f"{m}_{stat}" for m, stat in cv_summary.columns]
cv_summary = cv_summary.reset_index()
cv_summary.to_csv(RUN_DIR / "cv_summary_per_epoch.csv", index=False)

ranked = cv_summary.sort_values(
    [f"{m}_mean" for m in SELECT_METRICS], ascending=[m == "loss" for m in SELECT_METRICS], kind="mergesort"
)
assert int(ranked.iloc[0]["epoch"]) == BEST_EPOCH, "epoch chosen during training differs from the ranking"
best_row = cv_summary.set_index("epoch").loc[BEST_EPOCH]

print(f"Trained {EPOCHS_RUN} of up to {EPOCHS} epochs{' (stopped early)' if EPOCHS_RUN < EPOCHS else ''}")
print(f"Selected epoch {BEST_EPOCH} (mean over {N_FOLDS} folds):")
for m in SELECT_METRICS:
    print(f"  {m:12s} {best_row[f'{m}_mean']:.4f}  (std {best_row[f'{m}_std']:.4f})")
display(cv_summary.round(4))
display(cv_metrics[cv_metrics["epoch"] == BEST_EPOCH].round(4).reset_index(drop=True))

Trained 11 of up to 50 epochs (stopped early)
Selected epoch 5 (mean over 1 folds):
  hit@200      43.0000  (std nan)
  hit@100      29.0000  (std nan)
  hit@50       19.0000  (std nan)
  rocauc       0.8565  (std nan)


,epoch,hit@200_mean,hit@200_std,hit@100_mean,hit@100_std,hit@50_mean,hit@50_std,loss_mean,loss_std,rocauc_mean,rocauc_std,precision_mean,precision_std,recall_mean,recall_std,specificity_mean,specificity_std
0,0,27.0,NaN,17.0,NaN,10.0,NaN,3.4160,NaN,0.6612,NaN,0.0000,NaN,0.00,NaN,1.0000,NaN
1,1,42.0,NaN,30.0,NaN,16.0,NaN,2.9104,NaN,0.8048,NaN,0.1852,NaN,0.45,NaN,0.9208,NaN
2,2,34.0,NaN,22.0,NaN,16.0,NaN,2.4526,NaN,0.8116,NaN,0.0844,NaN,0.76,NaN,0.6700,NaN
3,3,40.0,NaN,22.0,NaN,13.0,NaN,2.0840,NaN,0.8195,NaN,0.0611,NaN,0.95,NaN,0.4160,NaN
4,4,39.0,NaN,24.0,NaN,16.0,NaN,2.0021,NaN,0.8338,NaN,0.1005,NaN,0.80,NaN,0.7136,NaN
5,5,43.0,NaN,29.0,NaN,19.0,NaN,1.8399,NaN,0.8565,NaN,0.1018,NaN,0.86,NaN,0.6964,NaN
6,6,27.0,NaN,19.0,NaN,13.0,NaN,1.9231,NaN,0.8091,NaN,0.0785,NaN,0.89,NaN,0.5820,NaN
7,7,25.0,NaN,19.0,NaN,13.0,NaN,2.7455,NaN,0.8329,NaN,0.1270,NaN,0.56,NaN,0.8460,NaN
8,8,33.0,NaN,21.0,NaN,10.0,NaN,2.7083,NaN,0.8202,NaN,0.1302,NaN,0.56,NaN,0.8504,NaN
9,9,28.0,NaN,14.0,NaN,5.0,NaN,2.8502,NaN,0.7788,NaN,0.1066,NaN,0.55,NaN,0.8156,NaN


,fold,epoch,hit@200,hit@100,hit@50,loss,rocauc,precision,recall,specificity
0,0,5,43.0,29.0,19.0,1.8399,0.8565,0.1018,0.86,0.6964


## 7. Selected models

Each fold's model from the selected epoch is `fold_<i>/best_model.ckpt` (saved during training).

In [9]:
best_models = [str(RUN_DIR / f"fold_{fold}" / "best_model.ckpt") for fold in range(N_FOLDS)]
best_folds = cv_metrics[cv_metrics["epoch"] == BEST_EPOCH].set_index("fold")
selection = dict(
    select_metrics=SELECT_METRICS,
    best_epoch=BEST_EPOCH,
    epochs_trained=EPOCHS_RUN,
    max_epochs=EPOCHS,
    patience=PATIENCE,
    stopped_early=EPOCHS_RUN < EPOCHS,
    n_folds=N_FOLDS,
    mean={m: float(best_row[f"{m}_mean"]) for m in metric_names},
    std={m: float(best_row[f"{m}_std"]) for m in metric_names},
    per_fold={m: {int(f): float(v) for f, v in best_folds[m].items()} for m in metric_names},
    models=best_models,
)
(RUN_DIR / "selection.json").write_text(json.dumps(selection, indent=2))
print(json.dumps(selection, indent=2))

{
  "select_metrics": [
    "hit@200",
    "hit@100",
    "hit@50",
    "rocauc"
  ],
  "best_epoch": 5,
  "epochs_trained": 11,
  "max_epochs": 50,
  "patience": 5,
  "stopped_early": true,
  "n_folds": 1,
  "mean": {
    "hit@200": 43.0,
    "hit@100": 29.0,
    "hit@50": 19.0,
    "loss": 1.8399208784103394,
    "rocauc": 0.856503963470459,
    "precision": 0.1017751470208168,
    "recall": 0.8600000143051147,
    "specificity": 0.696399986743927
  },
  "std": {
    "hit@200": NaN,
    "hit@100": NaN,
    "hit@50": NaN,
    "loss": NaN,
    "rocauc": NaN,
    "precision": NaN,
    "recall": NaN,
    "specificity": NaN
  },
  "per_fold": {
    "hit@200": {
      "0": 43.0
    },
    "hit@100": {
      "0": 29.0
    },
    "hit@50": {
      "0": 19.0
    },
    "loss": {
      "0": 1.8399208784103394
    },
    "rocauc": {
      "0": 0.856503963470459
    },
    "precision": {
      "0": 0.1017751470208168
    },
    "recall": {
      "0": 0.8600000143051147
    },
    "specificity": 

In [10]:
# Files produced by this run
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file():
        print(f"{str(path.relative_to(RUN_DIR)):60s} {path.stat().st_size / 1e6:8.2f} MB")

config.json                                                      0.00 MB
cv_metrics_per_epoch.csv                                         0.00 MB
cv_summary_per_epoch.csv                                         0.00 MB
fold_0/best_model.ckpt                                         338.59 MB
fold_0/data/data_train.csv                                       0.33 MB
fold_0/data/data_val.csv                                         0.11 MB
fold_0/metrics_per_epoch.csv                                     0.00 MB
progress.log                                                     0.00 MB
selection.json                                                   0.00 MB
